In [1]:
import os
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from ragas.llms import LangchainLLMWrapper
from langchain_deepseek import ChatDeepSeek
from ragas.dataset_schema import SingleTurnSample
from ragas.metrics import AnswerAccuracy
from datasets import load_dataset
from copy import deepcopy
import numpy as np
from scipy.stats import ttest_ind
import torch
import ast
load_dotenv()

/home/yhuang/ondemand/paper_clean/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
df = pd.read_csv('../QA_datasets_classified_qa_eval/output_csv/frames_UND_Gemini_Ragas.csv')
ragas_col = df['ragas_AA_short'].tolist()
pre_ragas = load_dataset("json", 
                        data_files="../QA_datasets_classified_qa_eval/intermediate/BASELINE_frames_UND_qa_Gemini_with_squad_scores.jsonl",
                        split="all")
assert len(pre_ragas) == len(ragas_col), "Fatal Error: length mismatch"

Generating train split: 440 examples [00:00, 6256.00 examples/s]


In [3]:
with_ragas = pre_ragas.add_column('ragas_AA_short', ragas_col)
with_ragas.to_json("./intermediate/frames_UND_Gemini_Ragas.jsonl", orient="records", lines=True)

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating json from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 30.42ba/s]


1973862

## Rewriting with GPT-4o
GPT-4o rewriting, then Gemini QA later

In [4]:
from helper_functions_qr import modification_in_batch, find_failed_rows_simple

In [5]:
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))
model = "gpt-4o-2024-11-20"
input_file = "./intermediate/frames_UND_Gemini_Ragas.jsonl"
output_file = "./intermediate/MODIFIED_frames_UND_Gemini_Ragas.jsonl"


In [6]:
# 清空输出文件（如果存在）
if os.path.exists(output_file):
    os.remove(output_file)
    print(f"Cleared existing output file: {output_file}")

# 处理所有样本（按批次）
question_modification = modification_in_batch(input_file, output_file, 'answer', client, model)

Total samples to process: 440
Batch size: 3


Processing batches:   3%|▎         | 5/147 [00:35<16:11,  6.84s/it]

Error processing sample 16: Invalid \escape: line 3 column 376 (char 652)


Processing batches:  20%|██        | 30/147 [03:15<12:28,  6.40s/it]

Error processing sample 92: Invalid \escape: line 3 column 95 (char 169)


...

Error processing sample 158: Invalid \escape: line 3 column 112 (char 330)


Processing batches:  54%|█████▎    | 79/147 [09:48<07:15,  6.41s/it]

Error processing sample 238: Invalid \escape: line 3 column 294 (char 443)


Processing batches: 100%|██████████| 147/147 [16:57<00:00,  6.92s/it]


All batch processing completed! Total processed: 440 samples
Results saved to: ./intermediate/MODIFIED_frames_UND_Gemini_Ragas.jsonl


In [7]:
find_failed_rows_simple(input_file, output_file)

=== 查找失败的行（简单方法）===
发现 0 个失败的行:


[]

In [8]:
df_view = pd.DataFrame(question_modification)
df_view

,original_question,modified_question,short_answer,model_original_answer,classifier_reasoning,original_f1,original_em,original_AA
0,How many years earlier would Punxsutawney Phil...,How many years earlier would Punxsutawney Phil...,87,"[The US Capitol is located in Washington D.C.,...",The query hinges on two critical pieces of inf...,0.0,0.0,0.0
1,"As of August 1, 2024, which country were holde...","As of August 1, 2024, which country was the FI...",France,[France],The query contains two distinct temporal refer...,1.0,1.0,1.0
2,What is the name of the vocalist from the firs...,What is the name of the vocalist for Meshuggah...,Jens Kidman,[Anders Fridén],"The query is complex and layered, requiring mu...",0.0,0.0,0.0
3,I have an element in mind and would like you t...,"Which element, named after a person, has an at...",Mendelevium is named after Dmitri Mendeleev.,[* Dmitri Mendeleev],"To solve this, first identify the scientist wh...",0.5,0.0,0.0
4,"As of Aug 3, 2024, the artist who released the...","As of August 3, 2024, DJ Khaled, the artist wh...",2,[4],The query links two entities (the artist of 'F...,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...
435,What is the name of the father of the first co...,"What is the name of Peter Treschow, who is ide...",Peter Treschow,[Unknown],The query involves tracing a complex genealogi...,0.0,0.0,0.0
436,Who was the winner of Tour de France the same ...,"Who won the Tour de France in 1955, the same y...",Louison Bobet,[* Louison Bobet],The query requires identifying a specific year...,1.0,1.0,1.0
437,A 2002 science fiction novel by an American au...,What is the name of the trilogy written by Nan...,The Sea of Trolls trilogy,"[Nalo Hopkinson, the author of *The Salt Roads...",The query provides specific details about the ...,0.1,0.0,0.0
438,Which movie musical produced a song that was i...,"Which movie musical, featuring the song ""Out H...",Fame,[* Ragtime],The query requires identifying a movie musical...,0.0,0.0,0.0


## Modified queries QA using Gemini-2.5-Flash

### Loading modified data

In [9]:
modified_set = load_dataset("json",
    data_files="./intermediate/MODIFIED_frames_UND_Gemini_Ragas.jsonl",
    split="all"  # 必须指定 split，否则默认返回 DatasetDict
)

# For issue resolution for QA, comment when there is no issue
#modified_set = modified_set.remove_columns(["model_new_answer"])

#modified_set.to_json(
    #"./intermediate/MODIFIED_frames_UND_Gemini_Ragas.jsonl",
    #orient="records",
    #lines=True
#)

Generating train split: 440 examples [00:00, 30273.35 examples/s]


### Implementation

In [10]:
from helper_functions_qr import (ask_short_answer, run_batch_shortQA_api, batch_QA_with_progress)
client = OpenAI(
    api_key=os.environ.get('GOOGLE_API_KEY'),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

In [15]:
modified_results = batch_QA_with_progress(
    modified_set,
    batch_fn=run_batch_shortQA_api,
    output_key="model_new_answer",
    fill_value=["error"],
    client=client,
    model="gemini-2.5-flash",
    temperature=0.0
)

Running model_new_answer: 100%|██████████| 44/44 [39:30<00:00, 53.88s/it]   


In [16]:
def has_mixed_types(seq):
    t = {type(x) for x in seq if x is not None}
    return len(t) > 1

for key, vals in modified_results.items():
    if has_mixed_types(vals):
        print(f"⚠️ Column '{key}' has mixed types, e.g., first 10 -> {vals[:10]}")

In [17]:
qa_modified = deepcopy(modified_set)
for key in modified_results:
    qa_modified = qa_modified.add_column(key, modified_results[key])

qa_modified.to_json("./intermediate/MODIFIED_frames_UND_Gemini_Ragas.jsonl", orient="records", lines=True)
df_qa_modified = pd.read_json("./intermediate/MODIFIED_frames_UND_Gemini_Ragas.jsonl", lines=True)
#df_qa_modified.to_csv('produced_files/modification_pilot_qa.csv')
df_qa_modified

Creating json from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 24.68ba/s]

,original_question,modified_question,short_answer,model_original_answer,classifier_reasoning,original_f1,original_em,original_AA,model_new_answer
0,How many years earlier would Punxsutawney Phil...,How many years earlier would Punxsutawney Phil...,87,"[The US Capitol is located in Washington D.C.,...",The query hinges on two critical pieces of inf...,0.0,0,0.0,[97 years]
1,"As of August 1, 2024, which country were holde...","As of August 1, 2024, which country was the FI...",France,[France],The query contains two distinct temporal refer...,1.0,1,1.0,[France]
2,What is the name of the vocalist from the firs...,What is the name of the vocalist for Meshuggah...,Jens Kidman,[Anders Fridén],"The query is complex and layered, requiring mu...",0.0,0,0.0,[Jens Kidman]
3,I have an element in mind and would like you t...,"Which element, named after a person, has an at...",Mendelevium is named after Dmitri Mendeleev.,[* Dmitri Mendeleev],"To solve this, first identify the scientist wh...",0.5,0,0.0,[Mendelevium]
4,"As of Aug 3, 2024, the artist who released the...","As of August 3, 2024, DJ Khaled, the artist wh...",2,[4],The query links two entities (the artist of 'F...,0.0,0,0.0,[2]
...,...,...,...,...,...,...,...,...,...
435,What is the name of the father of the first co...,"What is the name of Peter Treschow, who is ide...",Peter Treschow,[Unknown],The query involves tracing a complex genealogi...,0.0,0,0.0,[Peter Treschow]
436,Who was the winner of Tour de France the same ...,"Who won the Tour de France in 1955, the same y...",Louison Bobet,[* Louison Bobet],The query requires identifying a specific year...,1.0,1,1.0,[Louison Bobet]
437,A 2002 science fiction novel by an American au...,What is the name of the trilogy written by Nan...,The Sea of Trolls trilogy,"[Nalo Hopkinson, the author of *The Salt Roads...",The query provides specific details about the ...,0.1,0,0.0,[There is no trilogy written by Nancy Farmer t...
438,Which movie musical produced a song that was i...,"Which movie musical, featuring the song ""Out H...",Fame,[* Ragtime],The query requires identifying a movie musical...,0.0,0,0.0,[*Fame*]


## Evaluations

### Squad EM+F1

In [18]:
# Helper functions updated, RESTART!!
from helper_functions_qr import evaluate_squad_per_sample_multi_ref_pred

modified_set = load_dataset("json",
    data_files="./intermediate/MODIFIED_frames_UND_Gemini_Ragas.jsonl",
    split="all"  # 必须指定 split，否则默认返回 DatasetDict
)

qa_modified = deepcopy(modified_set)

Generating train split: 440 examples [00:00, 9370.32 examples/s]


In [19]:
squad_scored_modified, modified_f1_list, modified_em_list = evaluate_squad_per_sample_multi_ref_pred(qa_modified)
squad_scored_modified.to_json("./intermediate/MODIFIED_gpt4o_frames_UND_Gemini_new_squad.jsonl", orient="records", lines=True)

df = pd.read_json("./intermediate/MODIFIED_gpt4o_frames_UND_Gemini_new_squad.jsonl", lines=True)
df

Creating json from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 99.93ba/s]


,original_question,modified_question,short_answer,model_original_answer,classifier_reasoning,original_f1,original_em,original_AA,model_new_answer,new_em,new_f1
0,How many years earlier would Punxsutawney Phil...,How many years earlier would Punxsutawney Phil...,87,"[The US Capitol is located in Washington D.C.,...",The query hinges on two critical pieces of inf...,0.0,0,0.0,[97 years],0,0.000000
1,"As of August 1, 2024, which country were holde...","As of August 1, 2024, which country was the FI...",France,[France],The query contains two distinct temporal refer...,1.0,1,1.0,[France],1,1.000000
2,What is the name of the vocalist from the firs...,What is the name of the vocalist for Meshuggah...,Jens Kidman,[Anders Fridén],"The query is complex and layered, requiring mu...",0.0,0,0.0,[Jens Kidman],1,1.000000
3,I have an element in mind and would like you t...,"Which element, named after a person, has an at...",Mendelevium is named after Dmitri Mendeleev.,[* Dmitri Mendeleev],"To solve this, first identify the scientist wh...",0.5,0,0.0,[Mendelevium],0,0.285714
4,"As of Aug 3, 2024, the artist who released the...","As of August 3, 2024, DJ Khaled, the artist wh...",2,[4],The query links two entities (the artist of 'F...,0.0,0,0.0,[2],1,1.000000
...,...,...,...,...,...,...,...,...,...,...,...
435,What is the name of the father of the first co...,"What is the name of Peter Treschow, who is ide...",Peter Treschow,[Unknown],The query involves tracing a complex genealogi...,0.0,0,0.0,[Peter Treschow],1,1.000000
436,Who was the winner of Tour de France the same ...,"Who won the Tour de France in 1955, the same y...",Louison Bobet,[* Louison Bobet],The query requires identifying a specific year...,1.0,1,1.0,[Louison Bobet],1,1.000000
437,A 2002 science fiction novel by an American au...,What is the name of the trilogy written by Nan...,The Sea of Trolls trilogy,"[Nalo Hopkinson, the author of *The Salt Roads...",The query provides specific details about the ...,0.1,0,0.0,[There is no trilogy written by Nancy Farmer t...,0,0.126984
438,Which movie musical produced a song that was i...,"Which movie musical, featuring the song ""Out H...",Fame,[* Ragtime],The query requires identifying a movie musical...,0.0,0,0.0,[*Fame*],1,1.000000


In [20]:
modified_mean_em = np.mean(modified_em_list)  # em_scores: EM list per sample
modified_mean_f1 = np.mean(modified_f1_list)  # f1_scores F1 list per sample
print(f"New answers after modification Exact Match (avg): {modified_mean_em * 100:.2f}")
print(f"New answers after modification F1 Score (avg): {modified_mean_f1 * 100:.2f}")

original_em_list = qa_modified['original_em']
original_f1_list = qa_modified['original_f1']

original_mean_em = np.mean(original_em_list)  # em_scores: EM list per sample
original_mean_f1 = np.mean(original_f1_list)  # f1_scores F1 list per sample
print(f"Original answers Exact Match (avg): {original_mean_em * 100:.2f}")
print(f"Original answers F1 Score (avg): {original_mean_f1 * 100:.2f}")

f1_tstat, f1_pval = ttest_ind(modified_f1_list, original_f1_list, equal_var=False)
print(f"F1: t={f1_tstat:.3f}, p={f1_pval:.4f}")

em_tstat, em_pval = ttest_ind(modified_em_list, original_em_list, equal_var=False)
print(f"EM: t={em_tstat:.3f}, p={em_pval:.4f}")

New answers after modification Exact Match (avg): 29.77
New answers after modification F1 Score (avg): 46.48
Original answers Exact Match (avg): 24.55
Original answers F1 Score (avg): 37.08
F1: t=3.345, p=0.0009
EM: t=1.744, p=0.0815


### Ragas AA

In [2]:
from helper_functions_qr import answer_accuracy_modified
evaluator_llm = LangchainLLMWrapper(ChatDeepSeek(model="deepseek-chat", verbose=True, temperature=0))

In [3]:
squad_scored_modified = load_dataset("json",
    data_files="./intermediate/MODIFIED_gpt4o_frames_UND_Gemini_new_squad.jsonl",
    split="train")
result_with_AA = await answer_accuracy_modified(squad_scored_modified, evaluator_llm)
result_with_AA.to_csv("./output_csv/MODIFIED_gpt4o_frames_UND_Gemini_all_new_scores.csv")

Creating CSV from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 19.62ba/s]


562545

In [4]:
original_AA = list(result_with_AA["original_AA"])
modified_AA = list(result_with_AA["new_AA"])

original_mean_AA = np.mean(original_AA)
print(f"original AA (avg): {original_mean_AA * 100:.2f}")


modified_mean_AA = np.mean(modified_AA)
print(f"modified AA (avg): {modified_mean_AA * 100:.2f}")

AA_tstat, AA_pval = ttest_ind(modified_AA, original_AA, equal_var=False)
print(f"AA: t={AA_tstat:.3f}, p={AA_pval:.4f}")

original AA (avg): 42.05
modified AA (avg): 54.60
AA: t=3.911, p=0.0001


## Re-Classification

In [2]:
from helper_functions_qr import (
    batch_generate_responses_qwen3,
    get_judgments_from_responses,
    run_experiment,
    prepare_test_prompts)

print("HF_HOME:", os.environ.get("HF_HOME"))
print("HF_DATASETS_CACHE:", os.environ.get("HF_DATASETS_CACHE"))
print("HF_HUB_CACHE:", os.environ.get("HF_HUB_CACHE"))

HF_HOME: /scratch-local/yhuang/huggingface_cache
HF_DATASETS_CACHE: /scratch-local/yhuang/huggingface_cache/datasets
HF_HUB_CACHE: /scratch-local/yhuang/huggingface_cache/hub


### Loading data

In [3]:
reclassify_file = "./output_csv/MODIFIED_gpt4o_frames_UND_Gemini_all_new_scores.csv"
df_reclassify_file = pd.read_csv(reclassify_file)

### Loading model

In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM
Qwen3_4B = "Qwen/Qwen3-4B"
tokenizer = AutoTokenizer.from_pretrained(Qwen3_4B, padding_side='left')
model = AutoModelForCausalLM.from_pretrained(Qwen3_4B)
# 将模型移到可用设备
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
model = model.to(device)

Loading checkpoint shards: 100%|██████████| 3/3 [00:06<00:00,  2.00s/it]


cuda


### Prepare prompts

In [5]:
system_prompt = """
You are an expert analyst. Your task is to analyze and determine whether an input user query is "fully specified" or "underspecified".

"""

task_FS_UND = """
Analyze the following input user query:

{"query": "TARGET"}

Please provide your analysis in the following JSON format:

{"query": "TARGET", "reasoning": "[YOUR_DETAILED_REASONING]", "judgment": "[fully specified/underspecified]"}
"""

In [6]:
test_prompts = prepare_test_prompts(df_reclassify_file, task_FS_UND)
print(test_prompts[0])

Start preparing prompts...
# Testing data points: 440
Generation complete: 440 prompts
Average prompt length: 598 bytes (~149 tokens)

Analyze the following input user query:

{"query": "How many years earlier would Punxsutawney Phil have to be canonically alive to have made a Groundhog Day prediction in 1790, the year when the U.S. Capitol was temporarily located in Philadelphia, Pennsylvania?"}

Please provide your analysis in the following JSON format:

{"query": "How many years earlier would Punxsutawney Phil have to be canonically alive to have made a Groundhog Day prediction in 1790, the year when the U.S. Capitol was temporarily located in Philadelphia, Pennsylvania?", "reasoning": "[YOUR_DETAILED_REASONING]", "judgment": "[fully specified/underspecified]"}



In [7]:
test_df = run_experiment(tokenizer, model, test_prompts, system_prompt, df_reclassify_file)
test_df

100%|██████████| 88/88 [1:26:30<00:00, 58.99s/it] 


,original_question,modified_question,short_answer,model_original_answer,classifier_reasoning,original_f1,original_em,original_AA,model_new_answer,new_em,new_f1,new_AA,MODIFIED_thinking,MODIFIED_model_response,MODIFIED_model_pred
0,How many years earlier would Punxsutawney Phil...,How many years earlier would Punxsutawney Phil...,87,['The US Capitol is located in Washington D.C....,The query hinges on two critical pieces of inf...,0.0,0.0,0.0,['97 years'],0.0,0.000000,0.0,"<think>\nOkay, let's tackle this query step by...","{\n ""query"": ""How many years earlier would Pu...",underspecified
1,"As of August 1, 2024, which country were holde...","As of August 1, 2024, which country was the FI...",France,['France'],The query contains two distinct temporal refer...,1.0,1.0,1.0,['France'],1.0,1.000000,1.0,"<think>\nOkay, let me try to figure out if thi...","{\n ""query"": ""As of August 1, 2024, which cou...",fully specified
2,What is the name of the vocalist from the firs...,What is the name of the vocalist for Meshuggah...,Jens Kidman,['Anders Fridén'],"The query is complex and layered, requiring mu...",0.0,0.0,0.0,['Jens Kidman'],1.0,1.000000,1.0,"<think>\nOkay, let me try to figure out if thi...","{\n ""query"": ""What is the name of the vocal...",fully specified
3,I have an element in mind and would like you t...,"Which element, named after a person, has an at...",Mendelevium is named after Dmitri Mendeleev.,['* Dmitri Mendeleev'],"To solve this, first identify the scientist wh...",0.5,0.0,0.0,['Mendelevium'],0.0,0.285714,1.0,"<think>\nOkay, let's tackle this query step by...","{\n ""query"": ""Which element, named after a pe...",fully specified
4,"As of Aug 3, 2024, the artist who released the...","As of August 3, 2024, DJ Khaled, the artist wh...",2,['4'],The query links two entities (the artist of 'F...,0.0,0.0,0.0,['2'],1.0,1.000000,1.0,"<think>\nOkay, let's tackle this query step by...","{\n ""query"": ""As of August 3, 2024, DJ Khal...",underspecified
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
435,What is the name of the father of the first co...,"What is the name of Peter Treschow, who is ide...",Peter Treschow,['Unknown'],The query involves tracing a complex genealogi...,0.0,0.0,0.0,['Peter Treschow'],1.0,1.000000,1.0,"<think>\nOkay, let's tackle this query step by...","{\n ""query"": ""What is the name of Peter Tresc...",fully specified
436,Who was the winner of Tour de France the same ...,"Who won the Tour de France in 1955, the same y...",Louison Bobet,['* Louison Bobet'],The query requires identifying a specific year...,1.0,1.0,1.0,['Louison Bobet'],1.0,1.000000,1.0,"<think>\nOkay, let's see. The user is asking w...","{\n ""query"": ""Who won the Tour de France in 1...",fully specified
437,A 2002 science fiction novel by an American au...,What is the name of the trilogy written by Nan...,The Sea of Trolls trilogy,"['Nalo Hopkinson, the author of *The Salt Road...",The query provides specific details about the ...,0.1,0.0,0.0,['There is no trilogy written by Nancy Farmer ...,0.0,0.126984,0.0,"<think>\nOkay, let's break down this query ste...","{\n ""query"": ""What is the name of the trilogy...",fully specified
438,Which movie musical produced a song that was i...,"Which movie musical, featuring the song ""Out H...",Fame,['* Ragtime'],The query requires identifying a movie musical...,0.0,0.0,0.0,['*Fame*'],1.0,1.000000,1.0,"<think>\nOkay, let me try to figure this out. ...","{\n ""query"": ""Which movie musical, featuring ...",underspecified


In [8]:
test_df['MODIFIED_model_pred'].value_counts(normalize=True)


MODIFIED_model_pred
fully specified    0.706818
underspecified     0.293182
Name: proportion, dtype: float64

In [9]:
test_df['MODIFIED_model_pred'].value_counts()

MODIFIED_model_pred
fully specified    311
underspecified     129
Name: count, dtype: int64

In [10]:
test_df.to_csv('./output_csv/frames_UND_gpt4o_rewritten_reclassified.csv')